In [ ]:
# import libraries
from pylablib.devices import Thorlabs, Hubner
from PySide6.QtCore import Qt
from PySide6.QtWidgets import QApplication, QWidget, QMainWindow, QFileDialog, QLabel, QLineEdit, QCheckBox, QComboBox, QSpinBox, QDoubleSpinBox, QGridLayout, QVBoxLayout, QPushButton
from PIL import Image
import numpy as np
import time, gc

In [2]:
# define constant
COEFF = 512 * 67.49016
MAX_STEP = 25 * COEFF
DELTA_DIST = (1.52 - 1.00) / 1.52
GOAL_POSITION = [12.0, 24.0]

In [3]:
# control kinesis
def kinesis_move(stage, camera, realVelAccConverter, goalPosition, acc=2.0, maxVel=0.1):
    stage.setup_velocity(acceleration=acc * COEFF * realVelAccConverter[2], max_velocity=maxVel * COEFF * realVelAccConverter[1], scale=False)
    camera.setup_velocity(acceleration=acc * COEFF * realVelAccConverter[2] * DELTA_DIST, max_velocity=maxVel * COEFF * realVelAccConverter[1] * DELTA_DIST, scale=False)

    camGoalPosition = camera._get_position() / COEFF + (goalPosition - stage._get_position() / COEFF) * DELTA_DIST
    goalPosition_step = goalPosition * COEFF
    if goalPosition_step <= MAX_STEP:
        stage.move_to(goalPosition * COEFF)
        camera.move_to(camGoalPosition * COEFF)
    else:
        print("Overload! Set targetPosition <= 25.00 mm")
    return

In [4]:
# home kinesis
def kinesis_home(stage, camera, realVelAccConverter, stageHomePosition=1.0, cameraHomePosition=7.072, acc=2.0, maxVel=0.5):
    stage.setup_velocity(acceleration=acc * COEFF * realVelAccConverter[2], max_velocity=maxVel * COEFF * realVelAccConverter[1], scale=False)
    camera.setup_velocity(acceleration=acc * COEFF * realVelAccConverter[2] * DELTA_DIST, max_velocity=maxVel * COEFF * realVelAccConverter[1] * DELTA_DIST, scale=False)
    stage.move_to(stageHomePosition * COEFF)
    camera.move_to(cameraHomePosition * COEFF)
    return

In [5]:
# set emission filter
def FW_set(fw, fwPosition):
    fw.set_position(fwPosition)
    fw.close()
    return

In [6]:
# turn on laser
def LASER_ON(laser, line, intensity):
    intensity = intensity / 1000
    laser.query(str(line) + 'sla 1')
    laser.query(str(line) + '@cobasp ' + str(intensity))
    return

In [7]:
# turn off laser
def LASER_OFF(laser, line):
    laser.query(str(line) + 'sla 0')
    return

In [29]:
# acquire images
def acquireImages(Cam, exposure_ms, frameNum = 560):
    exposure_ms = exposure_ms / 1000
    Cam.open()
    counter = 0
    Cam.set_exposure(exposure_ms)
    Cam.setup_acquisition(nframes=frameNum)
    imgs = []
    Cam.start_acquisition()
    while True:
        Cam.wait_for_frame()
        frame = Cam.read_oldest_image()
        frame = np.round(frame/4094.0000001*255).astype('u1')    #convert 8-bit
        imgs.append(frame)
        frameNum -= 1
        if frameNum == 0:
            break
    Cam.stop_acquisition()
    print("done")
    Cam.close()
    return imgs

In [24]:
# save images
def saveMultiTIFF(data, fileName, od = 'C:/Users/PatchClampSystem/', t_max = None):
    frameNum = len(data)
    if t_max is None:
        images = [Image.fromarray(data[i]) for i in range(frameNum)]
        images[0].save(od + '/' + fileName + '.tif', save_all=True, append_images=images[1:], compression='tiff_deflate')
    elif t_max >= frameNum:
        images = [Image.fromarray(data[i]) for i in range(frameNum)]
        images[0].save(od + '/' + fileName + '.tif', save_all=True, append_images=images[1:], compression='tiff_deflate')
    else:
        temp = np.repeat(t_max, (frameNum // t_max))
        temp = np.append(temp, (frameNum % t_max))
        for j in range(int(np.ceil(frameNum / t_max))):
            j_max = (j + 1) * t_max
            if j_max > frameNum:
                j_max = frameNum
            j_min = j * t_max
            images = [Image.fromarray(data[i]) for i in range(j_min, j_max)]
            images[0].save(od + '/' + fileName + '_' + str(j + 1) + '.tif', save_all=True, append_images=images[1:])

In [2]:
# setup GUI
class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()

        labelVector = ["Emission filter", "Acquisition", "Excitation", "Intensity [mW]", "Exposure [msec]"]
        emFilterVector = ["535-40", "605-55", "690-50", "632-28"]

        self.setWindowTitle("Set Parameters")
        layout = QVBoxLayout()
        layout_odPath = QGridLayout()
        layout_ChunkSize = QGridLayout()
        layout_acqParams = QGridLayout()

        self.folder_path = ""
        self.odLabel = QLabel("Output Directory")
        self.od = QLineEdit(self)
        self.od.setMaxLength(1000)
        self.od.setText("C:/")
        self.refButton = QPushButton("Browse")
        self.refButton.pressed.connect(self.get_path)
        layout_odPath.addWidget(self.odLabel, 0, 0)
        layout_odPath.addWidget(self.od, 0, 1)
        layout_odPath.addWidget(self.refButton, 0, 2)
        layout.addLayout(layout_odPath)

        self.csLabel = QLabel("File Chunk Size")
        self.cs = QLineEdit(self)
        self.cs.setText("100")
        self.cs.setMaximumSize(75, 50)
        self.svLabel = QLabel("Sample Stage Velocity [mm / sec]")
        self.sv = QDoubleSpinBox(self)
        self.sv.setRange(0.001, 1)
        self.sv.setValue(0.10)
        self.sv.setSingleStep(0.10)
        self.sv.setMaximumSize(75, 50)
        self.ssLabel = QLabel("Cuvette Size")
        self.ss = QComboBox()
        self.ss.addItems(["10 mm", "20 mm"])
        self.emptyLabel = QLabel("                                              ")
        layout_ChunkSize.addWidget(self.csLabel, 0, 0)
        layout_ChunkSize.addWidget(self.cs, 0, 1)
        layout_ChunkSize.addWidget(self.emptyLabel, 0, 3)
        layout_ChunkSize.addWidget(self.svLabel, 1, 0)
        layout_ChunkSize.addWidget(self.sv, 1, 1)
        layout_ChunkSize.addWidget(self.ssLabel, 2, 0)
        layout_ChunkSize.addWidget(self.ss, 2, 1)
        layout_ChunkSize.addWidget(self.emptyLabel, 3, 2)
        layout_ChunkSize.addWidget(self.emptyLabel, 3, 3)
        layout.addLayout(layout_ChunkSize)

        self.labels = [QLabel(self) for i in range(5)]
        for i, label in enumerate(self.labels):
            label.setText(labelVector[i])
            layout_acqParams.addWidget(label, i, 0)

        self.emFilters = [QLabel(self) for i in range(4)]
        for i, emFilter in enumerate(self.emFilters):
            emFilter.setText(emFilterVector[i])
            layout_acqParams.addWidget(emFilter, 0, (i + 1))

        self.subFilters = [QLineEdit(self) for i in range(2)]
        for i, subFilter in enumerate(self.subFilters):
            subFilter.setPlaceholderText("Enter the filter info")
            subFilter.setMaxLength(12)
            layout_acqParams.addWidget(subFilter, 0, (i + len(emFilterVector) + 1))

        self.acqs = [QComboBox(self) for i in range(6)]
        for i, combobox in enumerate(self.acqs):
            combobox.addItems(["No", "Yes"])
            layout_acqParams.addWidget(combobox, 1, (i + 1))

        self.laserlines = [QComboBox(self) for i in range(6)]
        for i, combobox in enumerate(self.laserlines):
            combobox.addItems(["488 nm", "561 nm", "638 nm"])
            if i == 1:
                combobox.setCurrentIndex(1)
            elif i == 2:
                combobox.setCurrentIndex(2)
            elif i == 3:
                combobox.setCurrentIndex(1)
            layout_acqParams.addWidget(combobox, 2, (i + 1))

        self.dspinboxes = [QDoubleSpinBox(self) for i in range(6)]
        for i, dspinbox in enumerate(self.dspinboxes):
            dspinbox.setRange(0.0, 50.0)
            dspinbox.setValue(40.0)
            dspinbox.setSingleStep(0.10)
            layout_acqParams.addWidget(dspinbox, 3, (i + 1))

        self.spinboxes = [QSpinBox(self) for i in range(6)]
        for i, spinbox in enumerate(self.spinboxes):
            spinbox.setRange(33, 10000)
            spinbox.setValue(200)
            spinbox.setSingleStep(10)
            layout_acqParams.addWidget(spinbox, 4, (i + 1))

        self.PB1 = QPushButton("Cancel")
        self.PB1.pressed.connect(self.close)
        self.PB2 = QPushButton("OK")
        self.PB2.pressed.connect(self.return_values)
        self.PB2.released.connect(self.close)

        layout_acqParams.addWidget(self.PB1, 5, 5)
        layout_acqParams.addWidget(self.PB2, 5, 6)

        layout.addLayout(layout_acqParams)

        widget = QWidget()
        widget.setLayout(layout)

        self.setCentralWidget(widget)

    def get_path(self):
        path = QFileDialog.getExistingDirectory(self, 'Select Folder')
        if path:
            self.folder_path = path
            self.od.setText(self.folder_path)

    def select_switch(self):
        if self.ss1.clicked():
            self.ss1.setChecked(True)
            self.ss2.setChecked(False)
        elif self.ss2.clicked():
            self.ss1.setChecked(False)
            self.ss2.setChecked(True)

    def return_values(self):
        self.val_outputPath = self.od.text()
        self.val_ChunkSize = int(self.cs.text())
        self.val_StageVelocity = self.sv.value()
        self.val_SampleSize = self.ss.currentIndex()
        self.val_filter = []
        self.val_acqFlag = []
        self.val_laser = []
        self.val_intensity = []
        self.val_exposure = []
        for i in range(6):
            if i >= len(self.emFilters):
                self.val_filter.append(self.subFilters[i - len(self.emFilters)].text())
            else:
                self.val_filter.append(self.emFilters[i].text())

            if self.acqs[i].currentText() == "Yes":
                self.val_acqFlag.append(1)
            else:
                self.val_acqFlag.append(0)

            if self.laserlines[i].currentText() == "488 nm":
                self.val_laser.append(3)
            elif self.laserlines[i].currentText() == "561 nm":
                self.val_laser.append(1)
            else:
                self.val_laser.append(2)

            self.val_intensity.append(self.dspinboxes[i].value())
            self.val_exposure.append(self.spinboxes[i].value())


In [30]:
# connect devices
stage = Thorlabs.KinesisMotor('27270037')
time.sleep(0.5)
camera = Thorlabs.KinesisMotor('27269690')
time.sleep(0.5)
ThorCam = Thorlabs.TLCamera.ThorlabsTLCamera('31030')
time.sleep(0.5)
laser = Hubner.Cobolt('COM4')
time.sleep(0.5)
fw = Thorlabs.serial.FW('COM6')

ThorlabsError: query pcount? returned unexpected reply: b'Command error CMD_NOT_DEFINED'

In [ ]:
# display GUI
app = QApplication([])
window = MainWindow()
window.show()

app.exec()
print([window.val_outputPath, window.val_ChunkSize, window.val_StageVelocity, window.val_SampleSize, window.val_filter, window.val_acqFlag, window.val_laser, window.val_intensity, window.val_exposure])

In [ ]:
# run
realVelAccConverter = stage.get_scale()
acquisition = [i for i, x in enumerate(window.val_acqFlag) if x == 1]

kinesis_home(stage, camera, realVelAccConverter)
stage.wait_move()
camera.wait_move()
for acq in acquisition:
    FW_set(fw, fwPosition = (acq + 1))
    LASER_ON(laser, line = window.val_laser[acq], intensity = window.val_intensity[acq])

    kinesis_move(stage, camera, realVelAccConverter, goalPosition = GOAL_POSITION[window.val_SampleSize], maxVel = window.val_StageVelocity)
    stack = acquireImages(Cam = ThorCam, exposure_ms = window.val_exposure[acq])
    stage.wait_move()
    camera.wait_move()
    LASER_OFF(laser, line = window.val_laser[acq])
    kinesis_home(stage, camera, realVelAccConverter)
    saveMultiTIFF(data = stack, fileName = window.val_filter[acq], od = window.val_outputPath, t_max = window.val_ChunkSize)
    del stack
    gc.collect()

stage.move_to(0.0)
camera.move_to(0.0)
stage.wait_move()
camera.wait_move()
stage.close()
camera.close()
laser.query('l0')    # off laser
laser.close()
QApplication.shutdown(app)
print("If you want to restart, launch the coboltMonitor and press More>Restart")
print("Completed")

done


In [31]:
kinesis_home(stage, camera, realVelAccConverter)
stage.wait_move()
camera.wait_move()
stage.close()
camera.close()
#laser.query('l0')    # off laser
laser.close()
QApplication.shutdown(app)

In [31]:
fw = Thorlabs.serial.FW('COM6')

In [73]:
LASER_OFF(laser, line = window.val_laser[0])

In [32]:
del stack
gc.collect()

1504